# CASMI26 data tour: libraries, adducts, spectra

A compact, numbers-first tour of `train.parquet` (2.5 million MS/MS spectra) and the test file format. It is meant as a quick map before you build anything: where the spectra come from, how messy the metadata is, and how the training data differs from what you are scored on.

**Highlights**
- **2.54 M spectra of about 276 k structures** from **11 source libraries**. One library, `enveda-180`, holds 45% of the spectra and two thirds of the structures.
- **Instrument names are not normalised** (`Orbitrap` / `orbitrap`, `QTOF` / `qTof` / `qtof` ...). Map them to a few families before using them as a feature.
- **Adducts differ between train and test.** Training has many dimers (`[2M+Na]+`, `[2M+H]+`, ...); the visible test file has none.
- **Collision energy comes in several units** (eV, NCE, V, unknown).
- **A few metadata values are wildly off** (precursor errors of millions of ppm). Clip or filter them.

Only the competition data is used; everything below re-runs on CPU in a few minutes.

In [ ]:
import glob, os, re
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.width', 160)

def find_dir(pattern):
    hits = sorted(glob.glob(f'/kaggle/input/**/{pattern}', recursive=True), key=len)
    return os.path.dirname(hits[0]) if hits else 'data'
DATA_DIR = find_dir('train.parquet')
COLS = ['ingest_lib', 'inchikey14', 'molecular_formula', 'ionization_mode', 'instrument_type', 'adduct', 'adduct_orig',
        'precursor_mz', 'precursor_error_ppm', 'num_peaks', 'base_peak_intensity', 'collision_energy_orig_units']
t = pd.read_parquet(os.path.join(DATA_DIR, 'train.parquet'), columns=COLS)   # skip the heavy peak arrays
print(f'{len(t):,} spectra | {t.inchikey14.nunique():,} structures (InChIKey first block) | {t.ingest_lib.nunique()} libraries')
t.head(3)

## 1. Where the spectra come from

In [ ]:
lib = pd.DataFrame({'spectra': t.ingest_lib.value_counts(),
                    'structures': t.groupby('ingest_lib').inchikey14.nunique()}).sort_values('spectra', ascending=False)
lib['spectra_per_structure'] = (lib.spectra / lib.structures).round(1)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
lib.spectra.plot.barh(ax=ax[0], color='#4C72B0', logx=True, title='spectra per library (log scale)')
lib.structures.plot.barh(ax=ax[1], color='#55A868', logx=True, title='structures per library (log scale)')
for a in ax: a.invert_yaxis()
plt.tight_layout(); plt.show()
shared = t.groupby('inchikey14').ingest_lib.nunique()
print(f'structures found in more than one library: {(shared > 1).mean():.1%}')
lib

- `enveda-180` dominates: 1.15 M spectra (45%) of 183 k structures. `pluskal_ms2`, `riken` and `gnps` follow.
- Only about **10%** of structures appear in more than one library, so most structures have spectra from a single source and instrument.
- `enveda-np-examples` is tiny (250 structures). It comes from the same lab as the test data, which makes it tempting for validation. Remember that the public models in this competition were trained on all of `train.parquet`, so it is not a held-out set for them.

## 2. Instruments and ionization

In [ ]:
def instrument_family(s):
    s = str(s).lower()
    if s in ('nan', 'none', ''): return 'unknown'
    if 'timstof' in s: return 'timsTOF'
    if any(k in s for k in ('orbitrap', 'qft', 'ftms', 'itft', 'exactive', 'ft-icr', 'fticr', 'hybrid ft')): return 'Orbitrap / FT'
    if 'tof' in s: return 'Q-TOF / TOF'
    if any(k in s for k in ('trap', 'qit', 'lit', 'it-')): return 'ion trap'
    if any(k in s for k in ('qq', 'quattro', 'triple')): return 'triple quad'
    return 'other'
raw = t.instrument_type.value_counts(dropna=False)
print(f'{len(raw)} raw instrument labels; the 12 most common:')
print(raw.head(12).to_string())
t['instrument_family'] = t.instrument_type.map(instrument_family)
fam = pd.crosstab(t.ingest_lib, t.instrument_family)
fam.div(fam.sum(1), axis=0).plot.barh(stacked=True, figsize=(11, 4), colormap='tab10', title='instrument family by library (share of spectra)')
plt.legend(bbox_to_anchor=(1, 1)); plt.gca().invert_yaxis(); plt.tight_layout(); plt.show()
print('ionization:', t.ionization_mode.value_counts(normalize=True).round(3).to_dict())

- More than 80 raw labels collapse into a handful of families. Case and spelling vary (`Orbitrap` vs `orbitrap`, `QTOF` vs `qTof` vs `qtof`).
- `enveda-180` is all **timsTOF**, the same instrument family as the competition test data. Most public libraries are Orbitrap or Q-TOF.
- About **75%** of training spectra are positive mode.

## 3. Adducts: training vs the test file
The visible `test.parquet` is a small placeholder; the scored test is hidden. Its format and adduct mix are still a useful guide to what the hidden data looks like. We use it only for these counts.

In [ ]:
te = pd.read_parquet(os.path.join(DATA_DIR, 'test.parquet'), columns=['molecule_id', 'adduct', 'ionization_mode', 'instrument_type'])
mix = pd.DataFrame({'train': t.adduct.value_counts(normalize=True), 'visible test': te.adduct.value_counts(normalize=True)}).fillna(0)
mix = mix.sort_values('train', ascending=False).head(14)
mix.plot.bar(figsize=(12, 4), title='adduct share: train vs visible test'); plt.ylabel('share of spectra'); plt.tight_layout(); plt.show()
dimer = t.adduct.str.contains(r'\[2M|\[3M', regex=True)
print(f'train spectra with dimer/trimer adducts: {dimer.mean():.1%} | in the visible test: {te.adduct.str.contains(r"[23]M", regex=True).mean():.1%}')
print(f'train spectra whose adduct never appears in the visible test: {(~t.adduct.isin(set(te.adduct))).mean():.1%}')
print('visible test:', len(te), 'spectra,', te.molecule_id.nunique(), 'molecules,',
      te.groupby('molecule_id').size().describe()[['mean', '50%', 'max']].round(1).to_dict(), 'spectra per molecule')
print('visible test instruments:', te.instrument_type.value_counts().to_dict())
(mix * 100).round(1)

- The test side is mostly `[M+H]+` and `[M-H]-`, with no multimers. A sizeable share of training spectra (printed above) use adducts that never appear there (dimers, in-source losses ...). They are still useful library evidence, but if you train a model on spectra, consider weighting or filtering to the test-like adducts.
- Each test molecule comes with **several spectra** (about 3 on average). Combining the evidence across them per molecule is natural.

## 4. Spectra and metadata quality

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
ax[0].hist(np.log10(t.num_peaks.clip(lower=1)), bins=60, color='#4C72B0'); ax[0].set_title('peaks per spectrum (log10)')
e = t.precursor_error_ppm.dropna()
ax[1].hist(e.clip(-20, 20), bins=80, color='#C44E52'); ax[1].set_title('precursor error, ppm (clipped to +-20)')
ax[2].hist(t.precursor_mz.clip(0, 2000), bins=80, color='#55A868'); ax[2].set_title('precursor m/z')
plt.tight_layout(); plt.show()
print('peaks per spectrum:', t.num_peaks.describe()[['50%', 'mean', 'max']].round(0).to_dict())
print(f'precursor error: median {e.median():.2f} ppm, |error| > 20 ppm for {(e.abs() > 20).mean():.2%} of spectra, max {e.max():.3g} ppm')
print('collision-energy units:', t.collision_energy_orig_units.value_counts(dropna=False).to_dict())

- **Peaks:** the median spectrum has about 40 peaks, but the tail goes to tens of thousands (profile-like or noisy spectra). Keeping the top-k peaks per spectrum is a sensible default.
- **Precursor error:** most spectra sit within a few ppm. About 5% are off by more than 20 ppm, and some by far more than any instrument tolerance, which points to metadata mistakes. Clip or drop them before using precursor mass as a filter.
- **Collision energy** is recorded in eV, NCE, V or not at all. Treat energies from different units as different scales.

## 5. The structures

In [ ]:
first = t.drop_duplicates('inchikey14')
def element_counts(f):
    return {el: int(n) if n else 1 for el, n in re.findall(r'([A-Z][a-z]?)(\d*)', str(f))}
comp = pd.DataFrame([element_counts(f) for f in first.molecular_formula]).fillna(0)
share = (comp > 0).mean().sort_values(ascending=False)
print('share of structures containing each element:', (share[share > 0.001] * 100).round(1).to_dict())
spp = t.groupby('inchikey14').size()
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
ax[0].hist(comp.get('C', pd.Series(dtype=float)).clip(0, 80), bins=80, color='#8172B2'); ax[0].set_title('carbon atoms per structure')
ax[1].hist(np.log10(spp), bins=60, color='#937860'); ax[1].set_title('spectra per structure (log10)')
plt.tight_layout(); plt.show()
print('spectra per structure:', spp.describe()[['50%', 'mean', 'max']].round(1).to_dict())

- Nearly every structure has C, H and O, and most contain N. Fluorine in about one structure in five shows that training mixes natural products with many synthetic, drug-like compounds; the competition's test molecules come from biological extracts.
- The median structure has about 6 spectra (several adducts and energies), but a few reference compounds have more than a thousand. Down-weight them if you train on spectra, or they dominate.

## Takeaways
- Normalise instrument labels and collision-energy units before using them.
- The test side is `[M+H]+` / `[M-H]-` and several spectra per molecule; training is much more varied.
- Clip obviously broken metadata (precursor error) and very long spectra.
- For how the score is computed, including tautomers, stereo and the traps that crash a submission, see the companion notebook [CASMI26 Metric Explained: Tautomers, Stereo, Traps](https://www.kaggle.com/code/lakshyamewara0025/casmi26-metric-explained-tautomers-stereo-traps).

*Data: competition files only. Nothing here uses the visible test IDs beyond counting rows.*